<!-- DATA PROVIDER INSTRUCTIONS

1. Provide the name of your dataset, replacing the bracketed placeholder text.
2. Update the Registry of Open Data landing page URL, by replacing the bracketed placeholder text. The [REGISTRY_YAML_NAME] will correspond to the name of the YAML document in your pull request to the Registry of Open Data on Github, minus the .yaml file extension.
3. Remove these comment blocks when you have completed each section.

DATA PROVIDER INSTRUCTIONS -->

# Get to Know a Dataset: Collaborative Research in Computational Neuroscience

This notebook serves as a guided tour of the [Collaborative Research in Computational Neuroscience](https://registry.opendata.aws/crcnsarchive) dataset. More usage examples, tutorials, and documentation for this dataset and others can be found at the [Registry of Open Data on AWS](https://registry.opendata.aws/).

<!-- DATA PROVIDER INSTRUCTIONS

The goal of this section is to orient users to the structure of your dataset. 

1. How are key prefixes and objects organized in your S3 bucket?
2. What kinds of filetypes are represented in your dataset?
3. Explain with text what users are expected to encounter, and then demonstrate with code the organizational framework you applied when creating your dataset.
4. The responses to each question section are meant to be expanded or replaced as dictated by your dataset

DATA PROVIDER INSTRUCTIONS -->

### Q: How have you organized your dataset? Help us understand the key prefix structure of your S3 bucket.

The CRCNS.org archive consists of about 145 datasets.  At the top level of our s3 bucket, we will have a single key prefix "crcnsarchive" that in turn contains:

1. A Readme.txt document describing the archive.
2. Key prefixes for each of the individual datasets that are in the archive.  For example: crcnsarchive-hc-2, crcnsarchive-alm-1, crcnsarchive-pvc-1.
3. Key prefixes for contents of each of the individual datasets.  This will include a -License.txt file, a "-docs" prefix (documentation files), a "-data" prefix (for data files).

Full documentation about all the datasets can be found at: https://crcns.org/data-sets

This tutorial is about one of the datasets (hc-2), which is titled: "Multi-unit recordings from the rat hippocampus made during open field foraging."
Full documentation about this dataset is at: https://crcns.org/data-sets/hc/hc-2/about-hc-2.

When multiple datasets are added to the archive, this tutorial will probably be changed to a list of links to the tutorials made for individual datasets.



In [1]:
# CODING GUIDELINES FOR DATA PROVIDER
#
# General notebook coding guidelines:
# 1. Assume that your reader understands the basics of Jupyter Notebooks, Python, and their Python environment.
#    The focus of this tutorial is on your dataset.
# 2. For library requirements, list the required libraries in a comment block in "requirements.txt" format
#    (https://pip.pypa.io/en/stable/reference/requirements-file-format/)
# 3. Demonstrate importing libraries with the assumption that the user has correctly installed the required
#    libraries.
# 4. List and load all library dependencies once, at this point of the notebook, unless a complicated dependency
#    set makes it unweildy.
# 5. Remember, the goal of this tutorial is a 101-level introduction to your dataset using common tools and libraries.
#    Examples using specialized environments and deep-diving methods are better suited to follow-up tutorials.
#
# CODING GUIDELINES FOR DATA PROVIDER

### Tutorial for CRCNS hc-2 dataset:  Multi-unit recordings from the rat hippocampus made during open field foraging

First we will download and unpack the data files used for this notebook.  (This download is currently from crcns.org.  When the data is on s3, access to the data will be using s3).  To download and unpack the file, execute the following commands in a terminal.

In [ ]:
# download from crcns.org
! curl -O https://crcns.org/files/data/hc2/ec013.527.tar.gz
# unpack datafiles
! tar -xzf ec013.527.tar.gz

The above commands create a directory "crcns" with a subdirectory "hc2" and directory "ec013.527" within that.  Inside the "ec013.527" directory should be files:

```
ec013.527.clu.1		ec013.527.fet.3		ec013.527.mm.1		ec013.527.res.4		ec013.527.threshold.3
ec013.527.clu.2		ec013.527.fet.4		ec013.527.mm.2		ec013.527.spk.1		ec013.527.threshold.4
ec013.527.clu.3		ec013.527.led		ec013.527.mm.3		ec013.527.spk.2		ec013.527.whl
ec013.527.clu.4		ec013.527.m1m2.1	ec013.527.mm.4		ec013.527.spk.3		ec013.527.xml
ec013.527.eeg		ec013.527.m1m2.2	ec013.527.res.1		ec013.527.spk.4
ec013.527.fet.1		ec013.527.m1m2.3	ec013.527.res.2		ec013.527.threshold.1
ec013.527.fet.2		ec013.527.m1m2.4	ec013.527.res.3		ec013.527.threshold
```
These files are described in the data description document for the dataset at: https://crcns.org/files/data/hc2/crcns-hc2-data-description.pdf

Briefly, the numerical suffix (1-4) at the end of some file names indicates which "shank" (electrical sensor) the data is recorded from.  In this experiment, there were four shanks.  The files with "clu" in the name contain information about recorded data thought to represent "spikes" (action potentials) of neurons.  The "spk" files contain the recorded waveform (32 samples) associated with each spike from all 8 recording sites on the shank for which the spike was detected. 

### Goal of the notebook

The goal of this notebook is to display the waveforms recorded for each detected spike from all of the recording sites on one of the shanks.  To do that, we will need to read in the "clu" file, and use the position of cluster numbers in that file to get the rows in the "spk" file which has the spike waveforms.

In [3]:

# This notebook requires the following additional libraries
# (please install using the preferred method for your environment, e.g. pip, conda):
#
# matplotlib >= 3.10.8 (tested; earlier versions will probably work too)

# Import the libraries required for this notebook
# Built-ins
import numpy as np
# Installed libraries
import matplotlib.pyplot as plt


### Reading the "clu" file.

The data in the "clu" file consists of integers in ascii format, one integer per line.  The first integer is the number of "clusters" (patterns thought to represent neural spikes) found.  Each pattern (cluster) is assigned a number.  The numbers in the file after the first are cluster numbers.  The position of the number in the file (which line) indicates which rows in the corresponding "spk" file have the recorded waveform for the spike.  Cluster number 0 and 1 are noise and do not represent neural responses.

To read the data in this file (and later in other files) we first define the following function which reads a file containing numbers into a numpy 2-d array.


In [ ]:
def read_matrix_from_text_file(filename):
    matrix_list = []
    with open(filename, 'r') as file:
        for line in file:
            # Strip whitespace and split the line into individual number strings
            row_strings = line.strip().split()
            # Convert each string to a float and append to the list
            row_floats = [float(num) for num in row_strings]
            matrix_list.append(row_floats)
    
    # Convert the list of lists into a NumPy array
    return np.array(matrix_list)
print("Function read_matrix_from_text_file defined.")

Next we specify the shank number which will be stored in variable `shank`.  We will use shank number 2 in this example.  We then create the path to the "clu" file and load the file into variable `clu_data`.  The data is converted to integer (from float) since all the cluster numbers are integers.  Also, the 2-d array is converted to 1-d, since the data is 1-d anyway (only one number per line).

In [ ]:
shank = 2
session = "ec013.527"
base_path = f"crcns/hc2/{session}/"
clu_file_name = f"{session}.clu.{shank}"
clu_file_path = f"{base_path}/{clu_file_name}"
clu_data = read_matrix_from_text_file(clu_file_path).astype(int).ravel()
print(f"Number of rows read from file '{clu_file_name}' = " + str(len(clu_data)))


### Create list of cluster indicies

We next used the contents of the `clu_data` array to create a python dictionary `cluster_indicies` which maps each cluster number to a list of indicies (line number in the file) associated with that cluster.  Cluster numbers 0 and 1 are skipped because they are artifacts or noise and not neural spikes.

In [13]:
# get indicies of each cluster not 0 or 1
cluster_indicies = {}
index = -1
for id in clu_data[1:]:   # skip the first number (number of clusters)
    index += 1
    if id <= 1:
        # skip noise
        continue
    if id not in cluster_indicies:
        cluster_indicies[id] = []
    cluster_indicies[id].append(index)

To see what the loaded data looks like, we next display the number of spikes found (count of indicies) for each cluster (id) and up to the first 10 indicies of each of the spikes associated with the cluster.


In [ ]:
# display number of spikes found for each cluster
print("number of spikes found in each cluster and up to the first 10 indicies")
print("id\tcount\tfirst 10 indicies")
for id in sorted(cluster_indicies.keys()):
    num_found = len(cluster_indicies[id])
    print(f"{id}\t{num_found}\t{cluster_indicies[id][:10]}")

In order to display the waveform recorded for each spike, we need to load the data in the "spk" file.  That file is in binary format (2 byte, signed integers).  To load the data, the following function is defined.

In [20]:
# function to read binary data, two byte integers
def read_short_integers(filename, signed=True, big_endian=False):
    # Dtype string defines signed/unsigned and endianness
    if signed:
        dtype_str = '>i2' if big_endian else '<i2'
    else:
        dtype_str = '>u2' if big_endian else '<u2'
    
    # Read the file directly into a NumPy array with the specified dtype
    integers = np.fromfile(filename, dtype=dtype_str)
    return integers

We use the function defined above to load the spike waveforms.

In [ ]:
# use the function above to load the spike waveforms (from file .spk)
spk_file_name = f"{session}.spk.{shank}"
spk_file_path = f"{base_path}/{spk_file_name}"
spike_waveforms = read_short_integers(spk_file_path)
print(f"File '{spk_file_name}' loaded.  Size (number 2-byte integers) is " + str(len(spike_waveforms)) + ".")

The data in the `spk` file can be thought as being in a 2-d matrix, with each row containing the data for one spike waveform.  The width of each row is: 8 (recording sites on the shank) * 32 samples per waveform; or a total of 256 numbers per row (waveform).  Below we define a function to extract the waveform from a specified recordinig site (0-7) given the index to the complete waveform (recorded from all recording sites; that is, the indicies in the `cluster_indicies` dictionary).

In [ ]:
def get_spike_waveform(spike_waveforms, cluster_indicies, cluster_id, spike_number, recording_site):
    # returns waveform from spike_waveforms corresponding to cluster_indicies[cluster_id][spike_number] and recording_site
    # recording site is an integer from 0 to 7, which is the recording site on the shank
    # if no spike for the provided spike_number, return None
    if len(cluster_indicies[cluster_id]) <= spike_number:
        return None
    num_recording_sites = 8
    spike_width = 32 # number of samples per waveform recorded on one recording site
    spk_row_size = num_recording_sites * spike_width  # number samples per spike from all recording sites (multiple waveforms) 
    row = cluster_indicies[cluster_id][spike_number]
    start_index = row * spk_row_size + recording_site
    end_index = start_index + spk_row_size
    spike_waveform = spike_waveforms[start_index:end_index:num_recording_sites]
    return spike_waveform

Next we define a function to plot the waveforms for the first 20 spikes recorded for each cluster and recording site.

In [34]:
def plot_cluster_waveforms(spike_waveforms, cluster_indicies, cluster_id):
    # display first num_spikes_to_show spike waveforms, with the waveforms in each of the 8 recording sites shown in a separate plot
    num_spikes_to_show = 20
    width = 8 # width of plot area in inches
    height = 5 # height of plot area in inches
    fig, axes = plt.subplots(nrows=2, ncols=4, figsize=(width, height))
    num_plot_rows = 2
    num_plot_cols = 4
    for plt_row in range(num_plot_rows):
        for plt_col in range(num_plot_cols):
            recording_site = plt_row * num_plot_cols + plt_col  # should range from 0 to 7 if 8 total plots
            axes[plt_row, plt_col].set_title(f"Recording site {recording_site}")
            for spike_number in range(num_spikes_to_show):
                waveform = get_spike_waveform(spike_waveforms, cluster_indicies, cluster_id, spike_number, recording_site)
                if waveform is None:
                    break
                axes[plt_row, plt_col].plot(waveform)
    fig.suptitle(f"cluster {cluster_id}, " + str(spike_number + 1) + " spikes shown")
    plt.tight_layout(rect=[0, 0.3, 1, 0.95]) # adjust layout to prevent title overlap
    plt.show()

Finally, we use this to display the waveforms for each cluster.  A separate figure will appear for each cluster. 

In [ ]:
# make plots of clusters

for cluster_id in sorted(cluster_indicies.keys()):
    plot_cluster_waveforms(spike_waveforms, cluster_indicies, cluster_id)


The above figures show that for most spikes, only one or a few recording sites have a good detection of the spike waveform (displayed waveforms coincide).  This is because only a few recording sites (of the 8) are close to the neuron generating the spikes, and they pick up a stronger signal than the others.



### Part 2: display trace of animal movement

In this section we use another file in the dataset to generate a plot showing the movement of the animal (a rat) on a 2-d surface while the neural data was recorded.

The position of the animal is stored in the file with extension: ec013.527.whl.  Each row in the file has four number which indicate the position of two LEDs attached to the animal as recorded in a video (sampling rate is 39.06 Hz) taken during the experiment.  The first two numbers specify (x,y) coordinates, respectively, of the first LED, last two – of the second. -1 means
adequate tracking was not possible for these frames.

In the code below, the contents of the file are loaded into a numpy 2-d array.

In [ ]:
# load position data
whl_file_name = f"{base_path}/{session}.whl"
positions = read_matrix_from_text_file(whl_file_name)
print("positions is: " + str(positions) + ", shape is " + str(positions.shape))

Next, for simplicity, we only use one LED (using two provides the orientation of the animal's body).  Plus, we will remove all rows that have values of -1.  The result is stored in array `p2colf`.

In [ ]:
p2col = positions[:, 0:2] # have only first x and y
mask = positions[:, 0] != -1.
p2colf = p2col[mask]
print("p2colf is: " + str(p2colf) + ", shape is " + str(p2colf.shape))

Then, below, we use positon data in `p2colf` to plot the position of the animal (time trace of movement) during the experiment.

In [ ]:
x = p2colf[:,0]
y = p2colf[:,1]
time = np.arange(len(y)) * 0.01

fig, ax = plt.subplots(figsize=(10, 8))

# Plot the entire movement trajectory as a line
ax.plot(x, y, color='gray', linestyle='-', linewidth=1, label='Rat Trajectory')

# Overlay the specific events as colored circles
# ax.scatter(event_x, event_y, color='red', s=100, zorder=5, label='Neuron Firing Event')

# --- 3. Customize the plot ---

ax.set_title('Rat Movement Trajectory with Neuron Firing Events')
ax.set_xlabel('X Position')
ax.set_ylabel('Y Position')
ax.legend()
ax.grid(True)
# Ensure the aspect ratio is equal so the rectangular surface looks correct
ax.set_aspect('equal', adjustable='box') 

# Display the plot
plt.show()

<!-- DATA PROVIDER INSTRUCTIONS
This section is meant to orient users of your dataset to the formats present in your dataset, particularly if your dataset includes formats that may be unfamiliar to a general data scientist audience. This section should include:

1. Explanation of data format(s) (very common formats can be very briefly described, while less common
   or domain specific formats should include more explanation as well as links to official documentation)
2. Explanation of why the data format was chosen for your dataset
3. Recommendations around software and tooling to work with this data format
4. Explanation of any dataset-specific aspects to your usage of the format
5. Description of AWS services that may be useful to users working with your data
DATA PROVIDER INSTRUCTIONS -->

### Q: What data formats are present in your dataset? What kinds of data are stored using these formats? Can you give any advice for how you work with these data formats?

The most common format is MATLAB, but also Klusters (open source format for
neuroscience spiking data, what was used in the example above), HDF5, numpy, raw binary, and other formats.  The variety of data formats requires that good documentation be provided with each dataset.


<!-- DATA PROVIDER INSTRUCTIONS
The goal of this section is to demonstrate loading a portion of data from your dataset, and reveal something about its structure.
1. Load an object from S3
2. Show the structure of data in the object
DATA PROVIDER INSTRUCTIONS -->

### Q: Can you show us an example of downloading and loading data from your dataset?

An example was given above.  Although, the download location and perhaps method would change if the data is hosted at AWS.

<!-- DATA PROVIDER INSTRUCTIONS
The goal here is to visualize some aspect of your dataset in order to help users understand it. In addition to helping users of your dataset understand the dataset, an additional goal is to impress!

Please demonstrate any data preprocessing or reshaping required for your visualization(s).

https://www.reddit.com/r/dataisbeautiful/ for inspiration.
DATA PROVIDER INSTRUCTIONS -->

### Q: A picture is worth a thousand words. Show us a visual (or several!) from your dataset that either illustrates something informative about your dataset, or that you think might excite someone to dig in further.

Several examples were given above.

<!-- DATA PROVIDER INSTRUCTIONS
This section is less prescriptive / freeform than previous sections. The goal here is to show an opinionated example of answering a question using your data. The scale of your dataset may preclude a full example, and so feel free to limit the scope of this example (e.g. work on a subset of data). Users should be able to replicate your example in this notebook, and get a sense of how they would scale up.

A "toy" example is better than no example.

Ideally, your example would:
1. Transmit some of your domain & dataset experience to the reader, drawing on your own work as much as possible
2. Provide a jumping off point for users to extend your work, and do novel work of their own.

DATA PROVIDER INSTRUCTIONS -->

### Q: What is one question that you have answered using these data? Can you show us how you came to that answer?

The data was contributed to CRCNS.org by scientists at Rutgers University.  I personally can't show it (without additional study), but one of the papers published from the data, is titled: "Theta oscillations provide temporal windows for local circuit computation in the entorhinal-hippocampal loop".  (Mizuseki K, Sirota A, Pastalkova E, Buzsáki G., Neuron. 2009 Oct 29;64(2):267-80. (http://www.ncbi.nlm.nih.gov/pubmed/19874793)).

In additon, another paper that references the data (https://doi.org/10.12688/f1000research.3895.2) describe questions that could be answered and the value of the dataset:


> Several questions related to memory, navigation, spike time patterns, population coding, neuronal interactions, neuronal classification, replay, sleep homeostasis and oscillations have been studied based on this dataset6,21–43. However, this dataset may provide valuable information if subjected to yet further analyses. Improved spike sorting, neuron classification and more sophisticated analyses may extend and refine the initial conclusions and offer insights that were previously missed. For these reasons we provide both unprocessed (wide band) and processed versions of our data. In our experience, all methods have limitations and must undergo continuous revision.

> We believe that community-driven data sharing, cross-validation of data, unified data formats and large collaborative efforts will facilitate discovery and benefit future progress in neuroscience. We are well aware of the potential risks and arguments against data sharing but are of the opinion that the benefits of sharing countervail the risks.




<!-- DATA PROVIDER INSTRUCTIONS
This section is, like the previous one, intended to be freeform / non-prescriptive. The goal here is to provide a challenge to the community to do something novel with your dataset. That can either be novel in terms of the task, or novel in terms of methodological or computational approach.

Another way to consider this section, is as a wishlist. If you were less constrained by time, cost, skill, etc., what would you like to see achieved using these data? 

The challenge should, however, be somewhat realistic. A challenge that assumes e.g. original data collection, is likely to go unanswered.
DATA PROVIDER INSTRUCTIONS -->

### Q: What is one unanswered question that you think could be answered using these data? Do you have any recommendations or advice for someone wanting to answer this question?

I'm not an expert on the data, but see the answer to the question above for how this data could be used to address unanswered questions.


# DATA PROVIDER: PLEASE REMEMBER TO CLEAR ALL OUTPUTS BEFORE COMMITTING TO YOUR GITHUB REPOSITORY